In [ ]:
import hft
# import filegetter.filegetter as fgt

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
from scipy.stats import pearsonr, zscore,spearmanr



from hft.utils.wrapper import trade_to_depth
from hft.utils.validate import plot_stats
from hft.utils.target.mid_price_changes import all_return
from hft.utils.target import filled_return, mid_price_changes, mp_changes
from hft.utils.combine import linear_model, SGDlinear_model
from hft.utils.format import purged_train_test_split, single_split
from hft.utils.format import depth_to_depth

from hft.signal import arrive_rate, depth_age, depth_changes, fair_spread, large_jump, oir, price_distance, volume, weakoir, return_lag, order_flow 

ob=pd.read_csv('/Users/wook/Desktop/因子挖掘/processed_data_1s/BTCUSDT_book_snapshot_5_20240528_1s_orderbook.csv')
tr=pd.read_csv('/Users/wook/Desktop/因子挖掘/raw_data/BTCUSDT_trades_20240528.csv.gz')

In [ ]:
ob


In [ ]:
tr

In [ ]:
# 对于ob
ob = ob.rename(columns={
    'timestamp': 'ts',
    'bids[0].price': 'bp1', 'bids[0].amount': 'bv1',
    'asks[0].price': 'ap1', 'asks[0].amount': 'av1',
    'bids[1].price': 'bp2', 'bids[1].amount': 'bv2',
    'asks[1].price': 'ap2', 'asks[1].amount': 'av2',
    'bids[2].price': 'bp3', 'bids[2].amount': 'bv3',
    'asks[2].price': 'ap3', 'asks[2].amount': 'av3',
    'bids[3].price': 'bp4', 'bids[3].amount': 'bv4',
    'asks[3].price': 'ap4', 'asks[3].amount': 'av4',
    'bids[4].price': 'bp5', 'bids[4].amount': 'bv5',
    'asks[4].price': 'ap5', 'asks[4].amount': 'av5'
})

tr = tr.rename(columns={'timestamp':'ts', 'price':'p', 'amount':'v'})
tr['v'] = np.where(tr['side'] == 'sell', -tr['v'], tr['v'])

In [ ]:
ob.columns

In [ ]:
tr.columns

## target

In [ ]:
mid_price_ = (ob.ap1 + ob.bp1)/2
mid_price_.diff(600).shift(-600).fillna(0)    

### 因子1 bid_age ask_age

In [ ]:
import numpy as np
import pandas as pd
import numba as nb

@nb.jit
def get_age(x):
    last_value = x[-1]              #local_time
    age = 0
    for i in range(2, len(x)):
        if x[-i] != last_value:
            break
        age += 1
    return age

def bid_age(depth5, trade, n=600):
        bp1 = depth5['bp1']
        bp1_changes = bp1.rolling(n).apply(get_age, engine='numba', raw=True).fillna(0)     #numba用于优化提升代码速度，rolling.apply取窗口自己定义函数
        return bp1_changes

def aid_age(depth5, trade, n=600):
        ap1 = depth5['ap1']
        ap1_changes = ap1.rolling(n).apply(get_age, engine='numba', raw=True).fillna(0)
        return ap1_changes
    
bid_age = bid_age(depth5=ob, trade=tr, n=600)
aid_age = aid_age(depth5=ob, trade=tr, n=600)

pearsonr(bid_age, mid_price_.diff(600).shift(-600).fillna(0))



In [ ]:
pearsonr(aid_age, mid_price_.diff(600).shift(-600).fillna(0))

In [ ]:
plot_stats({"datas":{"orderbook":ob},"feature":bid_age})

In [ ]:
plot_stats({"datas":{"orderbook":ob},"feature":aid_age})

### 因子2：Arrive rate

In [ ]:
import numpy as np
import pandas as pd
from hft.utils.validate import plot_stats,MDI,numeric_stats,PFV

@trade_to_depth
def arrive_rate(depth5, trade, n=300):
    res = trade['ts'].diff(n).fillna(0) / n
    return res

ar=arrive_rate(depth5=ob,trade=tr,n=600)    #n代表取的交易数据量，600条

pearsonr(ar, mid_price_.diff(600).shift(-600).fillna(0))
ar_result = numeric_stats({"datas":{"orderbook":ob},"feature":ar})

In [ ]:
ar_result

In [ ]:
plot_stats({"datas":{"orderbook":ob},"feature":ar})

### 因子3：twap_diff

In [ ]:
from hft.utils.wrapper import trade_to_depth
import numpy as np
import pandas as pd

@trade_to_depth
def twap_diff(depth5, trade, n=600):
    return ((trade.v.abs() * trade.p).rolling(n).sum() / trade.v.abs().rolling(n).sum() - trade.p).fillna(0)

td=twap_diff(depth5=ob, trade=tr, n=600)
pearsonr(td, mid_price_.diff(600).shift(-600).fillna(0))

In [ ]:
td_result= numeric_stats({"datas":{"orderbook":ob},"feature":td})
td_result

In [ ]:
plot_stats({"datas":{"orderbook":ob},"feature":td})

### 因子4：cofi

In [ ]:
import numpy as np
import pandas as pd

def cofi(depth5, trade):
    
    a = depth5['bv1']*np.where(depth5['bp1'].diff()>=0,1,0)  # 买一价上涨时的买一量
    b = depth5['bv1'].shift()*np.where(depth5['bp1'].diff()<=0,1,0)  # 买一价下跌时的前期买一量
    c = depth5['av1']*np.where(depth5['ap1'].diff()<=0,1,0)  # 卖一价下跌时的卖一量  
    d = depth5['av1'].shift()*np.where(depth5['ap1'].diff()>=0,1,0)  # 卖一价上涨时的前期卖一量
    
    return (a - b - c + d).fillna(0)

co=cofi(depth5=ob, trade=tr,)
pearsonr(co, mid_price_.diff(600).shift(-600).fillna(0))

In [ ]:
co_result= numeric_stats({"datas":{"orderbook":ob},"feature":co})
co_result

In [ ]:
plot_stats({"datas":{"orderbook":ob},"feature":co})

### 因子5：oir_diff

In [ ]:
import numpy as np
import pandas as pd

def oir_diff(depth5, trade, length=10):
    
    a_volumes = depth5.av1 + depth5.av2 + depth5.av3 + depth5.av4 + depth5.av5 # 计算5档总卖单量
    b_volumes =  -(- depth5.bv1 - depth5.bv2 - depth5.bv3 - depth5.bv4 - depth5.bv5) # 计算5档总买单量
    # 用对数平滑极值影响，计算买单量占比（优势），在标准化到（-1，1）之间
    oiroir = (np.log(b_volumes + 1) / (np.log(b_volumes + 1) + np.log(a_volumes + 1)) - 0.5) * 2
    return pd.Series(oiroir.diff(10).fillna(0) - oiroir.diff().fillna(0)) #返回差分值

od=oir_diff(depth5=ob, trade=tr,)
pearsonr(od, mid_price_.diff(600).shift(-600).fillna(0))

In [ ]:
co_result= numeric_stats({"datas":{"orderbook":ob},"feature":od})
co_result

In [ ]:
plot_stats({"datas":{"orderbook":ob},"feature":od})

### 因子6：pmtv

In [ ]:
from hft.utils.wrapper import trade_to_depth
import numpy as np
import pandas as pd

@trade_to_depth
# 计算累计成交量
def volume(depth5, trade):
    return trade.v.cumsum()

def pmtv(depth5, trade):
    # 成交量变化量的绝对值 * 买一价变化量 用于确认价格趋势，较大时为正，大成交量推动价格上涨，涨跌信号
    #这是质量，不是能量（加速度）
    _pmtv = abs(volume(depth5=depth5, trade=trade).diff().fillna(0)) * depth5.bp1.diff().fillna(0)
    return _pmtv

pm=pmtv(depth5=ob, trade=tr,)
pearsonr(pm, mid_price_.diff(600).shift(-600).fillna(0))

In [ ]:
pm_result= numeric_stats({"datas":{"orderbook":ob},"feature":pm})
pm_result

In [ ]:
plot_stats({"datas":{"orderbook":ob},"feature":pm})

### 因子7：price_impact

In [ ]:
from hft.utils.wrapper import trade_to_depth
import numpy as np
import pandas as pd

def price_impact(depth5, trade, n=5):
    ask, bid, ask_v, bid_v = 0, 0, 0, 0
    # 计算个档位总额挂单金额和挂单总量
    for i in range(1, n+1):
        ask += depth5[f'ap{i}'] * depth5[f'av{i}']
        bid += depth5[f'bp{i}'] * depth5[f'bv{i}']
        ask_v += depth5[f'av{i}']
        bid_v += depth5[f'bv{i}']
    ask /= ask_v  #计算买卖加权平均价格
    bid /= bid_v
    # 卖方加权平均价-买方加权平均价，正值上涨，负值下跌
    return pd.Series(-(depth5['ap1'] - ask)/depth5['ap1'] - (depth5['bp1'] - bid)/depth5['bp1'], name="price_impact")

pi=price_impact(depth5=ob, trade=tr,)
pearsonr(pi, mid_price_.diff(600).shift(-600).fillna(0))

In [ ]:
pi_result= numeric_stats({"datas":{"orderbook":ob},"feature":pi})
pi_result

In [ ]:
plot_stats({"datas":{"orderbook":ob},"feature":pi})

### 因子8：weight price to mid

In [ ]:
from hft.utils.wrapper import trade_to_depth
import numpy as np
import pandas as pd

def get_columns(string,levels):
    res=[]
    for i in range(levels):
        res.append(string+'%s'%(i+1))
    
    return res
    
    
def weighted_price_to_mid(depth, trade, levels=5, alpha=1):
    avs = depth[get_columns("av", levels)] # 获取多档位数据
    bvs = depth[get_columns("bv", levels)]
    aps = depth[get_columns("ap", levels)]
    bps = depth[get_columns("bp", levels)]  
    if 0 < alpha < 1:  # 档位权重按照0.2的权重衰减
        decay_weights = np.array([alpha**n for n in range(levels)])
        avs *= decay_weights
        bvs *= decay_weights
    mp = (depth['ap1'] + depth['bp1'])/2  #计算中间价
    # 按照权重后的加权价格计算与中间价格的偏差
    return (avs.values * aps.values + bvs.values * bps.values).sum(axis=1) / (avs.values + bvs.values).sum(axis=1) - mp

wptm=weighted_price_to_mid(depth=ob, trade=tr,levels=5,alpha=0.2)
pearsonr(wptm, mid_price_.diff(600).shift(-600).fillna(0))

In [ ]:
wptm_result= numeric_stats({"datas":{"orderbook":ob},"feature":wptm})
wptm_result

In [ ]:
plot_stats({"datas":{"orderbook":ob},"feature":wptm})

### 因子9：withdraws

In [ ]:
import numba as nb
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

@nb.njit
def _ask_withdraws_volume(l, n, levels=5):  # 计算某个时刻各买方档位撤单量
    withdraws = 0.0
    for price_index in range(2, 2+4*levels, 4):
        now_p = n[price_index]
        for price_last_index in range(2, 2+4*levels, 4):
            if l[price_last_index] == now_p:
                withdraws -= min(n[price_index+1] - l[price_last_index + 1], 0.0)
    return withdraws

@nb.njit
def _bid_withdraws_volume(l, n, levels=5): # 计算某个时刻各卖方档位撤单量
    withdraws = 0.0
    for price_index in range(0, 4*levels, 4):
        now_p = n[price_index]
        for price_last_index in range(0, 4*levels, 4):
            if l[price_last_index] == now_p:
                withdraws -= min(n[price_index+1] - l[price_last_index + 1], 0.0)
    return withdraws

def ask_withdraws(depth5, trade): # 计算每个时刻的撤单量
    # 方案A：只选择数值列
    numeric_cols = depth5.select_dtypes(include=[np.number]).columns
    ob_values = depth5[numeric_cols].values.astype(np.float64)
    
    flows = np.zeros(len(ob_values), dtype=np.float64)
    for i in range(1, len(ob_values)):
        flows[i] = _ask_withdraws_volume(ob_values[i-1], ob_values[i])
    return pd.Series(flows, index=depth5.index)

def bid_withdraws(depth5, trade):
    # 方案A：只选择数值列
    numeric_cols = depth5.select_dtypes(include=[np.number]).columns
    ob_values = depth5[numeric_cols].values.astype(np.float64)
    
    flows = np.zeros(len(ob_values), dtype=np.float64)
    for i in range(1, len(ob_values)):
        flows[i] = _bid_withdraws_volume(ob_values[i-1], ob_values[i])
    return pd.Series(flows, index=depth5.index)

def withdraws_diff(depth5, trade):  #计算撤单量差值
    return ask_withdraws(depth5, trade) - bid_withdraws(depth5, trade)

# 使用代码
wd = withdraws_diff(depth5=ob, trade=td)
correlation = pearsonr(wd, mid_price_.diff(600).shift(-600).fillna(0))
print(f"相关性: {correlation}")


In [ ]:
wd_result= numeric_stats({"datas":{"orderbook":ob},"feature":wd})
wd_result
#plot_stats({"datas":{"orderbook":ob},"feature":od})

In [ ]:
plot_stats({"datas":{"orderbook":ob},"feature":wd})

### 因子10：big_vpin

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.style as mplstyle
from hft.utils.validate import plot_stats
import seaborn as sns
sns.set_style("darkgrid")


# 重置为原始函数
plt.style.use = plt.style.library.__getitem__

@trade_to_depth
def big_vpin(depth5,trade,n=600):
    return trade.v[::-1].rolling(n).sum().shift(-n).fillna(0)

bv=big_vpin(depth5=ob,trade=tr)
pearsonr(bv, mid_price_.diff(600).shift(-600).fillna(0))


In [ ]:
bv_result= numeric_stats({"datas":{"orderbook":ob},"feature":bv})
bv_result

In [ ]:
plot_stats({"datas":{"orderbook":ob},"feature":bv})

## 因子测试

##### 测试因子1：tpap_diff

In [ ]:
from hft.utils.wrapper import trade_to_depth
import numba as nb

@nb.jit
def get_average(x):
    cnt=0
    sum_x=0
    for i in range(len(x)):
        cnt+=1
        sum_x+=x[i]
    return sum_x/cnt

@trade_to_depth
def tpap_diff(depth5, trade, n=100):
    return (trade.p.rolling(n).apply(get_average,engine='numba',raw=True) - trade.p).fillna(0)

td=tpap_diff(depth5=ob,trade=tr,n=20)

mid_price = (ob.ap1 + ob.bp1)/2
print(pearsonr(td, mid_price.diff(500).shift(-500).fillna(0)))
print(spearmanr(td, mid_price.diff(500).shift(-500).fillna(0)))

plot_stats({"datas":{"orderbook":ob},"feature":td})

##### 测试因子2：pos_neg_diff

In [ ]:
from hft.utils.wrapper import trade_to_depth
import numba as nb

@nb.jit
def get_pos_neg_diff(x):
    pos = 0
    neg = 0
    for i in range(len(x)):
        if x[i]>=0:
            pos+=1
        else:
            neg+=1
    return pos-neg

@trade_to_depth
def pos_neg_diff(depth5, trade, n=100):
    return trade.v.rolling(n).apply(get_pos_neg_diff,engine='numba',raw=True).fillna(0)

pnd=pos_neg_diff(depth5=ob,trade=tr,n=15)

mid_price = (ob.ap1 + ob.bp1)/2
print(pearsonr(pnd, mid_price.diff(500).shift(-500).fillna(0)))
print(spearmanr(pnd, mid_price.diff(500).shift(-500).fillna(0)))

plot_stats({"datas":{"orderbook":ob},"feature":pnd})

##### 测试因子3：pos_neg_diff_p

In [ ]:
from hft.utils.wrapper import trade_to_depth
import numba as nb

@nb.jit
def get_pos_neg_diff_p(x):
    pos = 0
    neg = 0
    for i in range(len(x)):
        if x[i]>=0:
            pos+=x[i]
        else:
            neg+=x[i]
    return pos+neg

@trade_to_depth
def pos_neg_diff_p(depth5, trade, n=100):
    return (trade.v*trade.p).rolling(n).apply(get_pos_neg_diff_p,engine='numba',raw=True).fillna(0)

pndp=pos_neg_diff_p(depth5=ob,trade=tr,n=20)

mid_price = (ob.ap1 + ob.bp1)/2
print(pearsonr(pndp, mid_price.diff(500).shift(-500).fillna(0)))
print(spearmanr(pndp, mid_price.diff(500).shift(-500).fillna(0)))

plot_stats({"datas":{"orderbook":ob},"feature":pndp})

##### 测试因子4：pv_diff

In [ ]:
from hft.utils.wrapper import trade_to_depth

@trade_to_depth
def pv_diff(depth5, trade, n=100):
    return (trade.v*trade.p).diff(n).fillna(0)

pvd=pv_diff(depth5=ob,trade=tr,n=1)

mid_price = (ob.ap1 + ob.bp1)/2
print(pearsonr(pvd, mid_price.diff(500).shift(-500).fillna(0)))
print(spearmanr(pvd, mid_price.diff(500).shift(-500).fillna(0)))

plot_stats({"datas":{"orderbook":ob},"feature":pvd})

##### 测试因子5：tpap_diff_mean

In [ ]:
from hft.utils.wrapper import trade_to_depth
import numba as nb

@nb.jit
def get_average(x):
    cnt=0
    sum_x=0
    for i in range(len(x)):
        cnt+=1
        sum_x+=x[i]
    return sum_x/cnt

@trade_to_depth
def tpap_diff_average(depth5, trade, n=100):
    td=(trade.p.rolling(n).apply(get_average,engine='numba',raw=True) - trade.p).fillna(0)
    return td.rolling(n).apply(get_average,engine='numba',raw=True).fillna(0)

tda=tpap_diff_average(depth5=ob,trade=tr,n=15)

mid_price = (ob.ap1 + ob.bp1)/2
print(pearsonr(tda, mid_price.diff(500).shift(-500).fillna(0)))
print(spearmanr(tda, mid_price.diff(500).shift(-500).fillna(0)))

plot_stats({"datas":{"orderbook":ob},"feature":tda})

##### 测试因子6：tpap_mean_diff2

In [ ]:
from hft.utils.wrapper import trade_to_depth
import numba as nb

@nb.jit
def get_average(x):
    cnt=0
    sum_x=0
    for i in range(len(x)):
        cnt+=1
        sum_x+=x[i]
    return sum_x/cnt

@trade_to_depth
def tpap_mean_diff2(depth5, trade, n=100):
    td=(trade.p.rolling(n).apply(get_average,engine='numba',raw=True) - trade.p).fillna(0)
    return td-td.rolling(n).apply(get_average,engine='numba',raw=True).fillna(0)

tmd=tpap_mean_diff2(depth5=ob,trade=tr,n=30)

mid_price = (ob.ap1 + ob.bp1)/2
print(pearsonr(tmd, mid_price.diff(500).shift(-500).fillna(0)))
print(spearmanr(tmd, mid_price.diff(500).shift(-500).fillna(0)))

plot_stats({"datas":{"orderbook":ob},"feature":tmd})

##### 测试因子7：pv_diff_mean

In [ ]:
from hft.utils.wrapper import trade_to_depth
import numba as nb

@nb.jit
def get_average(x):
    cnt=0
    sum_x=0
    for i in range(len(x)):
        cnt+=1
        sum_x+=x[i]
    return sum_x/cnt

@trade_to_depth
def pv_diff_mean(depth5, trade, n=100):
    pd=(trade.v*trade.p).diff().fillna(0)
    return pd.rolling(n).apply(get_average,engine='numba',raw=True).fillna(0)

pdm=pv_diff_mean(depth5=ob,trade=tr,n=20)

mid_price = (ob.ap1 + ob.bp1)/2
print(pearsonr(pdm, mid_price.diff(500).shift(-500).fillna(0)))
print(spearmanr(pdm, mid_price.diff(500).shift(-500).fillna(0)))

plot_stats({"datas":{"orderbook":ob},"feature":pdm})

### 通过机器学习预测

In [ ]:
import statsmodels.api as sm
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold
from sklearn.metrics import f1_score
from collections import OrderedDict
from sklearn.metrics import mean_squared_error, r2_score
from sklearn import linear_model
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.linear_model import LogisticRegression

In [ ]:
# 计算各种特征
td = twap_diff(depth5=ob, trade=tr, n=20)
pndp = pos_neg_diff_p(depth5=ob, trade=tr, n=20)
pvd = pv_diff(depth5=ob, trade=tr, n=1)
tda = tpap_diff_average(depth5=ob, trade=tr, n=15)
pdm = pv_diff_mean(depth5=ob, trade=tr, n=20)

# 计算中间价
mid_price = (ob.ap1 + ob.bp1) / 2

# 创建特征DataFrame
df_td = pd.DataFrame(list(td), columns=['td'])
df_pndp = pd.DataFrame(list(pndp), columns=['pndp'])
df_pvd = pd.DataFrame(list(pvd), columns=['pvd'])
df_tda = pd.DataFrame(list(tda), columns=['tda'])
df_pdm = pd.DataFrame(list(pdm), columns=['pdm'])

# 合并特征
X = pd.concat([df_td, df_pndp, df_pvd, df_tda, df_pdm], axis=1)

# 计算目标变量 - 方法1：指定列名
y = pd.DataFrame(mid_price.diff(500).shift(-500).fillna(0), columns=['mid_price'])

# 添加常数项
X = sm.add_constant(X)

# 将连续的价格变化转换为分类标签
y.loc[y['mid_price'] > 0, 'mid_price'] = 1   # 价格上涨
y.loc[y['mid_price'] < 0, 'mid_price'] = -1  # 价格下跌
y.loc[y['mid_price'] == 0, 'mid_price'] = 0  # 价格不变

# 建立OLS回归模型
est = sm.OLS(y, X)
model = est.fit()
print(model.summary())

# 替代方案：使用Logistic回归进行分类预测
print("\n" + "="*50)
print("Logistic回归结果：")
print("="*50)

# 为logistic回归准备二分类标签 (上涨=1, 不上涨=0)
y_binary = (y['mid_price'] > 0).astype(int)

# 建立Logistic回归模型
logit_model = sm.Logit(y_binary, X)
logit_result = logit_model.fit()
print(logit_result.summary())

# 预测和评估
y_pred_proba = logit_result.predict(X)
y_pred = (y_pred_proba > 0.5).astype(int)

# 计算准确率
accuracy = (y_pred == y_binary).mean()
print(f"\n模型准确率: {accuracy:.4f}")

# 计算F1分数（如果有正负样本）
if len(np.unique(y_binary)) > 1:
    f1 = f1_score(y_binary, y_pred)
    print(f"F1分数: {f1:.4f}")

# 特征重要性分析
print(f"\n特征系数 (Logistic回归):")
feature_names = X.columns
coefficients = logit_result.params
for name, coef in zip(feature_names, coefficients):
    print(f"{name}: {coef:.6f}")

# 可视化预测结果
plt.figure(figsize=(12, 8))

plt.subplot(2, 2, 1)
plt.plot(mid_price.index[:1000], mid_price.iloc[:1000])
plt.title('中间价走势')
plt.xlabel('时间')
plt.ylabel('价格')

plt.subplot(2, 2, 2)
plt.hist(y['mid_price'], bins=50, alpha=0.7)
plt.title('价格变化分布')
plt.xlabel('价格变化')
plt.ylabel('频次')

plt.subplot(2, 2, 3)
plt.scatter(y_pred_proba, y_binary, alpha=0.5)
plt.xlabel('预测概率')
plt.ylabel('实际标签')
plt.title('预测概率 vs 实际标签')

plt.subplot(2, 2, 4)
confusion_data = pd.crosstab(y_binary, y_pred, margins=True)
print(f"\n混淆矩阵:")
print(confusion_data)

plt.tight_layout()
plt.show()

# 交叉验证评估
print(f"\n" + "="*50)
print("交叉验证结果:")
print("="*50)

from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression

# 使用sklearn的LogisticRegression进行交叉验证
sklearn_model = LogisticRegression(max_iter=1000)
cv_scores = cross_val_score(sklearn_model, X, y_binary, cv=5, scoring='accuracy')

print(f"5折交叉验证准确率: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

## LightGBM

In [ ]:
import lightgbm as lgb
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt

# 1. 检查原始数据分布
print("=== 原始数据分析 ===")
print("目标变量唯一值:", np.unique(y))
print("目标变量分布:")
value_counts = pd.Series(y).value_counts().sort_index()
print(value_counts)
print("各类别占比:")
print(pd.Series(y).value_counts(normalize=True).sort_index())

# 2. 标签转换：(-1, 0, 1) -> (0, 1, 2)
def convert_labels_for_lgb(y):
    """将标签从(-1, 0, 1)转换为(0, 1, 2)"""
    y_converted = y.copy()
    y_converted[y == -1] = 0  # 下跌
    y_converted[y == 0] = 1   # 不变  
    y_converted[y == 1] = 2   # 上涨
    return y_converted.astype(int)

def convert_labels_back(y_pred):
    """将预测结果从(0, 1, 2)转换回(-1, 0, 1)"""
    y_back = y_pred.copy()
    y_back[y_pred == 0] = -1  # 下跌
    y_back[y_pred == 1] = 0   # 不变
    y_back[y_pred == 2] = 1   # 上涨
    return y_back

# 转换标签
y_converted = convert_labels_for_lgb(y)
print(f"\n转换后的标签唯一值: {np.unique(y_converted)}")

# 3. 数据分割（使用分层抽样保持类别比例）
X_train, X_test, y_train, y_test = train_test_split(
    X, y_converted, 
    test_size=0.3, 
    random_state=42, 
    stratify=y_converted  # 分层抽样
)

print(f"\n=== 数据分割结果 ===")
print(f"训练集: {X_train.shape[0]} 样本")
print(f"测试集: {X_test.shape[0]} 样本")
print("训练集标签分布:")
print(pd.Series(y_train).value_counts().sort_index())
print("测试集标签分布:")
print(pd.Series(y_test).value_counts().sort_index())

# 4. 创建LightGBM数据集
lgb_train = lgb.Dataset(X_train, y_train)
lgb_eval = lgb.Dataset(X_test, y_test, reference=lgb_train)

# 5. 设置分类参数
params = {
    'objective': 'multiclass',
    'num_class': 3,
    'metric': 'multi_logloss',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.1,  # 稍微提高学习率
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': 1,
    'random_state': 42,
    'min_data_in_leaf': 20,  # 防止过拟合
    'min_gain_to_split': 0.1,  # 防止过拟合
}

# 6. 设置回调函数
early_stopping_callback = lgb.early_stopping(stopping_rounds=20)
verbose_callback = lgb.log_evaluation(period=50)

print(f"\n=== 开始训练分类模型 ===")

# 7. 训练模型
try:
    gbm = lgb.train(
        params,
        lgb_train,
        num_boost_round=1000,
        valid_sets=[lgb_train, lgb_eval],
        valid_names=['train', 'eval'],
        callbacks=[early_stopping_callback, verbose_callback]
    )
    
    print(f"模型训练成功！最佳迭代次数: {gbm.best_iteration}")
    
except Exception as e:
    print(f"训练出错: {e}")
    # 如果还有问题，检查数据
    print("检查训练数据:")
    print(f"X_train shape: {X_train.shape}")
    print(f"y_train unique: {np.unique(y_train)}")
    print(f"y_train min: {y_train.min()}, max: {y_train.max()}")

# 8. 预测和评估
if 'gbm' in locals():
    # 预测概率
    y_pred_proba = gbm.predict(X_test, num_iteration=gbm.best_iteration)
    
    # 转换为类别预测
    y_pred_lgb = np.argmax(y_pred_proba, axis=1)
    
    # 转换回原始标签格式
    y_pred_original = convert_labels_back(y_pred_lgb)
    y_test_original = convert_labels_back(y_test)
    
    print(f"\n=== 模型评估结果 ===")
    
    # 计算准确率
    accuracy = accuracy_score(y_test_original, y_pred_original)
    print(f"准确率: {accuracy:.4f}")
    
    # 详细分类报告
    print(f"\n分类报告:")
    target_names = ['下跌(-1)', '不变(0)', '上涨(1)']
    print(classification_report(y_test_original, y_pred_original, 
                              target_names=target_names, digits=4))
    
    # 混淆矩阵
    cm = confusion_matrix(y_test_original, y_pred_original, labels=[-1, 0, 1])
    print(f"\n混淆矩阵:")
    print("实际\\预测   下跌   不变   上涨")
    for i, (actual_label, row_name) in enumerate(zip([-1, 0, 1], ['下跌', '不变', '上涨'])):
        print(f"{row_name:4s}    {cm[i,0]:6d} {cm[i,1]:6d} {cm[i,2]:6d}")
    
    # 特征重要性
    feature_importance = gbm.feature_importance(importance_type='gain')
    feature_names = X.columns if hasattr(X, 'columns') else [f'feature_{i}' for i in range(X.shape[1])]
    
    print(f"\n特征重要性排序:")
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': feature_importance
    }).sort_values('importance', ascending=False)
    print(importance_df)
    
    # 9. 可视化结果
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # 特征重要性图
    sorted_idx = np.argsort(feature_importance)
    axes[0,0].barh(range(len(feature_importance)), feature_importance[sorted_idx])
    axes[0,0].set_yticks(range(len(feature_importance)))
    axes[0,0].set_yticklabels(np.array(feature_names)[sorted_idx])
    axes[0,0].set_title('特征重要性')
    axes[0,0].set_xlabel('重要性分数')
    
    # 混淆矩阵热图
    im = axes[0,1].imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    axes[0,1].set_title('混淆矩阵')
    tick_marks = np.arange(3)
    axes[0,1].set_xticks(tick_marks)
    axes[0,1].set_yticks(tick_marks)
    axes[0,1].set_xticklabels(['下跌', '不变', '上涨'])
    axes[0,1].set_yticklabels(['下跌', '不变', '上涨'])
    axes[0,1].set_ylabel('实际标签')
    axes[0,1].set_xlabel('预测标签')
    
    # 在混淆矩阵中添加数字
    for i in range(3):
        for j in range(3):
            axes[0,1].text(j, i, cm[i, j], ha="center", va="center", color="red")
    
    # 预测概率分布
    axes[1,0].hist(y_pred_proba[:, 0], alpha=0.5, label='下跌概率', bins=30)
    axes[1,0].hist(y_pred_proba[:, 1], alpha=0.5, label='不变概率', bins=30)  
    axes[1,0].hist(y_pred_proba[:, 2], alpha=0.5, label='上涨概率', bins=30)
    axes[1,0].set_title('预测概率分布')
    axes[1,0].set_xlabel('预测概率')
    axes[1,0].set_ylabel('频次')
    axes[1,0].legend()
    
    # 各类别的预测置信度
    max_proba = np.max(y_pred_proba, axis=1)
    axes[1,1].hist(max_proba, bins=30, alpha=0.7)
    axes[1,1].set_title('预测置信度分布')
    axes[1,1].set_xlabel('最大预测概率')
    axes[1,1].set_ylabel('频次')
    axes[1,1].axvline(np.mean(max_proba), color='red', linestyle='--', 
                     label=f'平均置信度: {np.mean(max_proba):.3f}')
    axes[1,1].legend()
    
    plt.tight_layout()
    plt.show()
    
    # 10. 性能分析
    print(f"\n=== 性能分析 ===")
    
    # 基准对比
    most_frequent_class = pd.Series(y_test_original).value_counts().index[0]
    baseline_accuracy = (y_test_original == most_frequent_class).mean()
    print(f"基线准确率（预测众数）: {baseline_accuracy:.4f}")
    print(f"模型改善: {(accuracy - baseline_accuracy):.4f}")
    
    # 各类别的预测表现
    for class_label, class_name in zip([-1, 0, 1], ['下跌', '不变', '上涨']):
        mask = (y_test_original == class_label)
        if mask.sum() > 0:
            class_acc = (y_pred_original[mask] == class_label).mean()
            print(f"{class_name}类别预测准确率: {class_acc:.4f}")
    
    # 平均预测置信度
    print(f"平均预测置信度: {np.mean(max_proba):.4f}")
    print(f"高置信度预测比例(>0.6): {(max_proba > 0.6).mean():.4f}")

## 线性回归

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn import linear_model, metrics
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# 合并特征
X = pd.concat([ar, co, od, pi, wptm, wd, bv, td, pndp, tda, pdm], axis=1)

# 处理列名问题 - 将所有列名转换为字符串
X.columns = X.columns.astype(str)
print(f"特征数量: {X.shape[1]}")
print(f"特征列名: {list(X.columns)}")

# 创建目标变量
y = pd.DataFrame(mid_price.diff(500).shift(-500).fillna(0))
y.columns = ['mid_price']

# 将目标变量转换为二分类 (上涨=1, 不上涨=0)
y_binary = y.copy()
y_binary['mid_price'] = (y['mid_price'] > 0).astype(int)

print(f"数据形状: X={X.shape}, y={y_binary.shape}")
print(f"目标变量分布:")
print(y_binary['mid_price'].value_counts())
print(f"上涨比例: {y_binary['mid_price'].mean():.3f}")

# 检查数据中的无穷大和NaN值
print(f"\n数据质量检查:")
print(f"X中的NaN数量: {X.isnull().sum().sum()}")
print(f"X中的无穷大数量: {np.isinf(X.select_dtypes(include=[np.number])).sum().sum()}")
print(f"y中的NaN数量: {y_binary.isnull().sum().sum()}")

# 清理数据
# 替换无穷大值
X = X.replace([np.inf, -np.inf], np.nan)
# 填充NaN值
X = X.fillna(X.median())
y_binary = y_binary.fillna(0)

print(f"\n清理后数据质量:")
print(f"X中的NaN数量: {X.isnull().sum().sum()}")
print(f"y中的NaN数量: {y_binary.isnull().sum().sum()}")

# 10折交叉验证
kf = KFold(n_splits=10, shuffle=True, random_state=42)
acc_scores = []
mse_scores = []
r2_scores = []

print(f"\n开始10折交叉验证...")

fold = 1
for train_split, test_split in kf.split(X, y_binary):
    print(f"处理第 {fold} 折...")
    
    # 分割数据
    X_train, X_test = X.iloc[train_split], X.iloc[test_split]
    y_train, y_test = y_binary.iloc[train_split], y_binary.iloc[test_split]
    
    # 特征标准化
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # 训练线性回归模型
    model = linear_model.LinearRegression()
    model.fit(X_train_scaled, y_train)
    
    # 预测
    y_pred_continuous = model.predict(X_test_scaled)
    
    # 将连续预测转换为二分类 (>0.5为上涨，<=0.5为不上涨)
    y_pred_binary = (y_pred_continuous > 0.5).astype(int)
    
    # 计算指标
    mse = metrics.mean_squared_error(y_test, y_pred_continuous)
    r2 = metrics.r2_score(y_test, y_pred_continuous)
    accuracy = metrics.accuracy_score(y_test, y_pred_binary)
    
    mse_scores.append(mse)
    r2_scores.append(r2)
    acc_scores.append(accuracy)
    
    print(f"  第{fold}折 - 准确率: {accuracy:.4f}, MSE: {mse:.6f}, R²: {r2:.6f}")
    fold += 1

# 计算平均性能
print(f"\n=== 10折交叉验证结果 ===")
print(f"平均准确率: {np.mean(acc_scores):.4f} (±{np.std(acc_scores):.4f})")
print(f"平均MSE: {np.mean(mse_scores):.6f} (±{np.std(mse_scores):.6f})")
print(f"平均R²: {np.mean(r2_scores):.6f} (±{np.std(r2_scores):.6f})")

# 详细结果
print(f"\n各折详细结果:")
for i in range(10):
    print(f"第{i+1}折: 准确率={acc_scores[i]:.4f}, MSE={mse_scores[i]:.6f}, R²={r2_scores[i]:.6f}")

# 可视化结果
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 准确率分布
axes[0,0].bar(range(1, 11), acc_scores, alpha=0.7, color='skyblue')
axes[0,0].axhline(y=np.mean(acc_scores), color='red', linestyle='--', 
                 label=f'平均值: {np.mean(acc_scores):.4f}')
axes[0,0].set_xlabel('折数')
axes[0,0].set_ylabel('准确率')
axes[0,0].set_title('各折准确率')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

# MSE分布
axes[0,1].bar(range(1, 11), mse_scores, alpha=0.7, color='lightcoral')
axes[0,1].axhline(y=np.mean(mse_scores), color='red', linestyle='--',
                 label=f'平均值: {np.mean(mse_scores):.6f}')
axes[0,1].set_xlabel('折数')
axes[0,1].set_ylabel('MSE')
axes[0,1].set_title('各折MSE')
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

# R²分布
axes[1,0].bar(range(1, 11), r2_scores, alpha=0.7, color='lightgreen')
axes[1,0].axhline(y=np.mean(r2_scores), color='red', linestyle='--',
                 label=f'平均值: {np.mean(r2_scores):.6f}')
axes[1,0].set_xlabel('折数')
axes[1,0].set_ylabel('R²')
axes[1,0].set_title('各折R²分数')
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)

# 性能指标箱线图
metrics_data = [acc_scores, mse_scores, r2_scores]
metric_names = ['准确率', 'MSE', 'R²']
axes[1,1].boxplot(metrics_data, labels=metric_names)
axes[1,1].set_title('性能指标分布')
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 训练最终模型（使用全部数据）
print(f"\n=== 最终模型训练 ===")
scaler_final = StandardScaler()
X_scaled_final = scaler_final.fit_transform(X)

final_model = linear_model.LinearRegression()
final_model.fit(X_scaled_final, y_binary)

# 特征系数分析
feature_coefs = pd.DataFrame({
    'feature': X.columns,
    'coefficient': final_model.coef_[0],
    'abs_coefficient': np.abs(final_model.coef_[0])
}).sort_values('abs_coefficient', ascending=False)

print(f"特征重要性（按系数绝对值排序）:")
print(feature_coefs.to_string(index=False))

# 模型诊断
y_pred_final = final_model.predict(X_scaled_final)
final_mse = metrics.mean_squared_error(y_binary, y_pred_final)
final_r2 = metrics.r2_score(y_binary, y_pred_final)
final_acc = metrics.accuracy_score(y_binary, (y_pred_final > 0.5).astype(int))

print(f"\n最终模型（全数据）性能:")
print(f"准确率: {final_acc:.4f}")
print(f"MSE: {final_mse:.6f}")
print(f"R²: {final_r2:.6f}")

# 基准对比
baseline_accuracy = max(y_binary['mid_price'].value_counts(normalize=True))
print(f"\n基准准确率（预测众数）: {baseline_accuracy:.4f}")
print(f"模型改善: {np.mean(acc_scores) - baseline_accuracy:.4f}")

if np.mean(acc_scores) > baseline_accuracy + 0.01:
    print("✅ 模型性能显著优于基准")
else:
    print("⚠️ 模型性能接近基准，可能需要改进特征或模型")

## 逻辑回归

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn import linear_model, metrics
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# 合并特征
X = pd.concat([ar, co, od, pi, wptm, wd, bv, td, pndp, tda, pdm], axis=1)

# 修复列名类型问题 - 将所有列名转换为字符串
X.columns = X.columns.astype(str)
print(f"特征数量: {X.shape[1]}")
print(f"特征列名: {list(X.columns)}")

# 创建目标变量
y = pd.DataFrame(mid_price.diff(500).shift(-500).fillna(0))
y.columns = ['mid_price']

# 将目标变量转换为二分类 (上涨=1, 不上涨=0)
y_binary = y.copy()
y_binary['mid_price'] = (y['mid_price'] > 0).astype(int)

print(f"数据形状: X={X.shape}, y={y_binary.shape}")
print(f"目标变量分布:")
print(y_binary['mid_price'].value_counts())
print(f"上涨比例: {y_binary['mid_price'].mean():.3f}")

# 数据清理
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median())
y_binary = y_binary.fillna(0)

print(f"清理后数据质量:")
print(f"X中的NaN数量: {X.isnull().sum().sum()}")
print(f"y中的NaN数量: {y_binary.isnull().sum().sum()}")

# 10折交叉验证
kf = KFold(n_splits=10, shuffle=True, random_state=42)
acc_scores = []
precision_scores = []
recall_scores = []
f1_scores = []
auc_scores = []

print(f"\n开始10折交叉验证 (逻辑回归)...")

fold = 1
all_y_test = []
all_y_pred = []
all_y_pred_proba = []

for train_split, test_split in kf.split(X, y_binary):
    print(f"处理第 {fold} 折...")
    
    # 分割数据
    X_train, X_test = X.iloc[train_split], X.iloc[test_split]
    y_train, y_test = y_binary.iloc[train_split], y_binary.iloc[test_split]
    
    # 特征标准化（逻辑回归对特征尺度敏感）
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # 训练逻辑回归模型
    model = linear_model.LogisticRegression(
        max_iter=1000,  # 增加最大迭代次数
        random_state=42,
        solver='liblinear'  # 对小数据集效果好
    )
    model.fit(X_train_scaled, y_train.values.ravel())  # 使用.values.ravel()确保是1D数组
    
    # 预测
    y_pred = model.predict(X_test_scaled)
    y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]  # 获取正类概率
    
    # 计算各种指标
    accuracy = metrics.accuracy_score(y_test, y_pred)
    precision = metrics.precision_score(y_test, y_pred)
    recall = metrics.recall_score(y_test, y_pred)
    f1 = metrics.f1_score(y_test, y_pred)
    auc = metrics.roc_auc_score(y_test, y_pred_proba)
    
    acc_scores.append(accuracy)
    precision_scores.append(precision)
    recall_scores.append(recall)
    f1_scores.append(f1)
    auc_scores.append(auc)
    
    # 保存用于最终分析
    all_y_test.extend(y_test.values.ravel())
    all_y_pred.extend(y_pred)
    all_y_pred_proba.extend(y_pred_proba)
    
    print(f"  第{fold}折 - 准确率: {accuracy:.4f}, 精确率: {precision:.4f}, 召回率: {recall:.4f}, F1: {f1:.4f}, AUC: {auc:.4f}")
    fold += 1

# 计算平均性能
print(f"\n=== 10折交叉验证结果 ===")
print(f"平均准确率: {np.mean(acc_scores):.4f} (±{np.std(acc_scores):.4f})")
print(f"平均精确率: {np.mean(precision_scores):.4f} (±{np.std(precision_scores):.4f})")
print(f"平均召回率: {np.mean(recall_scores):.4f} (±{np.std(recall_scores):.4f})")
print(f"平均F1分数: {np.mean(f1_scores):.4f} (±{np.std(f1_scores):.4f})")
print(f"平均AUC: {np.mean(auc_scores):.4f} (±{np.std(auc_scores):.4f})")

# 转换为numpy数组
all_y_test = np.array(all_y_test)
all_y_pred = np.array(all_y_pred)
all_y_pred_proba = np.array(all_y_pred_proba)

# 整体混淆矩阵
cm = confusion_matrix(all_y_test, all_y_pred)
print(f"\n=== 整体混淆矩阵 ===")
print("实际\\预测   不涨    上涨")
print(f"不涨      {cm[0,0]:6d} {cm[0,1]:6d}")
print(f"上涨      {cm[1,0]:6d} {cm[1,1]:6d}")

# 分类报告
print(f"\n=== 详细分类报告 ===")
print(classification_report(all_y_test, all_y_pred, target_names=['不上涨', '上涨']))

# 可视化结果
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# 1. 各折准确率
axes[0,0].bar(range(1, 11), acc_scores, alpha=0.7, color='skyblue')
axes[0,0].axhline(y=np.mean(acc_scores), color='red', linestyle='--', 
                 label=f'平均: {np.mean(acc_scores):.4f}')
axes[0,0].set_xlabel('折数')
axes[0,0].set_ylabel('准确率')
axes[0,0].set_title('各折准确率')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

# 2. AUC分布
axes[0,1].bar(range(1, 11), auc_scores, alpha=0.7, color='lightgreen')
axes[0,1].axhline(y=np.mean(auc_scores), color='red', linestyle='--',
                 label=f'平均: {np.mean(auc_scores):.4f}')
axes[0,1].set_xlabel('折数')
axes[0,1].set_ylabel('AUC')
axes[0,1].set_title('各折AUC')
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

# 3. F1分数分布
axes[0,2].bar(range(1, 11), f1_scores, alpha=0.7, color='lightcoral')
axes[0,2].axhline(y=np.mean(f1_scores), color='red', linestyle='--',
                 label=f'平均: {np.mean(f1_scores):.4f}')
axes[0,2].set_xlabel('折数')
axes[0,2].set_ylabel('F1分数')
axes[0,2].set_title('各折F1分数')
axes[0,2].legend()
axes[0,2].grid(True, alpha=0.3)

# 4. 混淆矩阵热图
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1,0])
axes[1,0].set_title('混淆矩阵')
axes[1,0].set_xlabel('预测标签')
axes[1,0].set_ylabel('实际标签')

# 5. ROC曲线
from sklearn.metrics import roc_curve
fpr, tpr, thresholds = roc_curve(all_y_test, all_y_pred_proba)
axes[1,1].plot(fpr, tpr, color='darkorange', lw=2, 
               label=f'ROC曲线 (AUC = {np.mean(auc_scores):.4f})')
axes[1,1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
axes[1,1].set_xlim([0.0, 1.0])
axes[1,1].set_ylim([0.0, 1.05])
axes[1,1].set_xlabel('假正率')
axes[1,1].set_ylabel('真正率')
axes[1,1].set_title('ROC曲线')
axes[1,1].legend(loc="lower right")

# 6. 预测概率分布
axes[1,2].hist(all_y_pred_proba[all_y_test==0], bins=50, alpha=0.5, 
               label='实际不涨', color='red', density=True)
axes[1,2].hist(all_y_pred_proba[all_y_test==1], bins=50, alpha=0.5, 
               label='实际上涨', color='blue', density=True)
axes[1,2].set_xlabel('预测上涨概率')
axes[1,2].set_ylabel('密度')
axes[1,2].set_title('预测概率分布')
axes[1,2].legend()
axes[1,2].axvline(x=0.5, color='black', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

# 训练最终模型
print(f"\n=== 最终模型训练 ===")
scaler_final = StandardScaler()
X_scaled_final = scaler_final.fit_transform(X)

final_model = linear_model.LogisticRegression(
    max_iter=1000,
    random_state=42,
    solver='liblinear'
)
final_model.fit(X_scaled_final, y_binary.values.ravel())

# 特征系数分析
feature_coefs = pd.DataFrame({
    'feature': X.columns,
    'coefficient': final_model.coef_[0],
    'abs_coefficient': np.abs(final_model.coef_[0])
}).sort_values('abs_coefficient', ascending=False)

print(f"特征重要性（按系数绝对值排序）:")
print(feature_coefs.to_string(index=False))

# 基准对比
print(f"\n=== 性能对比 ===")
baseline_accuracy = max(y_binary['mid_price'].value_counts(normalize=True))
print(f"基准准确率（预测众数）: {baseline_accuracy:.4f}")
print(f"逻辑回归平均准确率: {np.mean(acc_scores):.4f}")
print(f"模型改善: {np.mean(acc_scores) - baseline_accuracy:.4f}")

if np.mean(acc_scores) > baseline_accuracy + 0.01:
    print("✅ 逻辑回归模型性能显著优于基准")
    print(f"AUC={np.mean(auc_scores):.4f} 说明模型有一定的区分能力")
else:
    print("⚠️ 模型性能接近基准，可能需要改进特征或尝试其他算法")

# 预测置信度分析
high_confidence = np.sum((all_y_pred_proba > 0.7) | (all_y_pred_proba < 0.3))
print(f"\n=== 预测置信度分析 ===")
print(f"高置信度预测比例 (>0.7 或 <0.3): {high_confidence/len(all_y_pred_proba):.3f}")
print(f"平均预测概率: {np.mean(all_y_pred_proba):.4f}")

# 不同置信度下的准确率
for threshold in [0.6, 0.7, 0.8, 0.9]:
    mask = (all_y_pred_proba > threshold) | (all_y_pred_proba < (1-threshold))
    if np.sum(mask) > 0:
        high_conf_acc = np.mean(all_y_test[mask] == all_y_pred[mask])
        print(f"置信度>{threshold}时的准确率: {high_conf_acc:.4f} (样本数: {np.sum(mask)})")

## 逻辑回归+向前逐步选择

In [ ]:
import itertools
import pandas as pd
import numpy as np
import statsmodels.api as sm
import sklearn.metrics as metrics
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

def optimized_forward_stepwise(y, X, max_features=None, scoring='f1', cv_folds=5, 
                              early_stopping=True, tolerance=0.001, verbose=True):
    """
    优化的前向逐步特征选择
    
    参数:
    - y: 目标变量
    - X: 特征矩阵 
    - max_features: 最大特征数量限制
    - scoring: 评分标准 ('f1', 'accuracy', 'roc_auc')
    - cv_folds: 交叉验证折数
    - early_stopping: 是否启用早停
    - tolerance: 早停容忍度
    - verbose: 是否显示详细信息
    
    返回:
    - df_results: 包含每步结果的DataFrame
    - best_features: 最佳特征组合
    """
    
    # 初始化
    selected_features = []
    remaining_features = list(X.columns)
    
    # 结果存储
    results = {
        'step': [],
        'selected_feature': [],
        'features_list': [],
        'n_features': [],
        f'{scoring}_score': [],
        f'{scoring}_cv_mean': [],
        f'{scoring}_cv_std': [],
        'aic': [],
        'bic': [],
        'improvement': []
    }
    
    # 设置最大特征数
    if max_features is None:
        max_features = min(len(remaining_features), 20)  # 限制最大特征数
    
    if verbose:
        print(f"开始前向逐步特征选择...")
        print(f"总特征数: {len(remaining_features)}")
        print(f"最大选择特征数: {max_features}")
        print(f"评分标准: {scoring}")
        print("-" * 60)
    
    best_score = -np.inf
    no_improvement_count = 0
    
    # 逐步添加特征
    for step in range(1, max_features + 1):
        if len(remaining_features) == 0:
            break
            
        step_best_score = -np.inf
        step_best_feature = None
        step_best_results = None
        
        if verbose:
            print(f"\n第 {step} 步: 从 {len(remaining_features)} 个候选特征中选择...")
        
        # 尝试每个剩余特征
        for candidate_feature in remaining_features:
            # 当前特征组合
            current_features = selected_features + [candidate_feature]
            X_current = X[current_features]
            
            try:
                # 使用statsmodels进行logistic回归 (获取AIC/BIC)
                X_with_const = sm.add_constant(X_current)
                model_sm = sm.Logit(y, X_with_const).fit(disp=0)  # disp=0 关闭输出
                
                # 预测和评分
                y_pred_proba = model_sm.predict(X_with_const)
                y_pred_class = (y_pred_proba > 0.5).astype(int)
                
                # 计算主要评分
                if scoring == 'f1':
                    main_score = metrics.f1_score(y, y_pred_class)
                elif scoring == 'accuracy':
                    main_score = metrics.accuracy_score(y, y_pred_class)
                elif scoring == 'roc_auc':
                    main_score = metrics.roc_auc_score(y, y_pred_proba)
                else:
                    raise ValueError(f"不支持的评分标准: {scoring}")
                
                # 交叉验证评分 (使用sklearn的LogisticRegression以提高速度)
                lr_model = LogisticRegression(max_iter=1000, random_state=42)
                cv_scores = cross_val_score(lr_model, X_current, y.values.ravel(), 
                                          cv=cv_folds, scoring=scoring)
                cv_mean = cv_scores.mean()
                cv_std = cv_scores.std()
                
                # 存储当前候选特征的结果
                candidate_results = {
                    'feature': candidate_feature,
                    'main_score': main_score,
                    'cv_mean': cv_mean,
                    'cv_std': cv_std,
                    'aic': model_sm.aic,
                    'bic': model_sm.bic
                }
                
                # 更新最佳结果 (使用交叉验证分数)
                if cv_mean > step_best_score:
                    step_best_score = cv_mean
                    step_best_feature = candidate_feature
                    step_best_results = candidate_results
                    
            except Exception as e:
                if verbose:
                    print(f"   跳过特征 {candidate_feature}: {str(e)}")
                continue
        
        # 检查是否找到了改进
        if step_best_feature is None:
            if verbose:
                print("   没有找到有效的特征，停止搜索")
            break
            
        # 计算改进幅度
        improvement = step_best_score - best_score if step > 1 else step_best_score
        
        # 早停检查
        if early_stopping and step > 1 and improvement < tolerance:
            no_improvement_count += 1
            if no_improvement_count >= 2:  # 连续2步没有显著改进就停止
                if verbose:
                    print(f"   改进幅度 {improvement:.6f} < 容忍度 {tolerance}")
                    print("   触发早停，结束搜索")
                break
        else:
            no_improvement_count = 0
        
        # 更新选择的特征
        selected_features.append(step_best_feature)
        remaining_features.remove(step_best_feature)
        best_score = step_best_score
        
        # 记录结果
        results['step'].append(step)
        results['selected_feature'].append(step_best_feature)
        results['features_list'].append(selected_features.copy())
        results['n_features'].append(len(selected_features))
        results[f'{scoring}_score'].append(step_best_results['main_score'])
        results[f'{scoring}_cv_mean'].append(step_best_results['cv_mean'])
        results[f'{scoring}_cv_std'].append(step_best_results['cv_std'])
        results['aic'].append(step_best_results['aic'])
        results['bic'].append(step_best_results['bic'])
        results['improvement'].append(improvement)
        
        if verbose:
            print(f"   ✓ 选择特征: {step_best_feature}")
            print(f"   ✓ {scoring.upper()}分数: {step_best_results['main_score']:.6f}")
            print(f"   ✓ CV平均分数: {step_best_results['cv_mean']:.6f} (±{step_best_results['cv_std']:.6f})")
            print(f"   ✓ AIC: {step_best_results['aic']:.2f}, BIC: {step_best_results['bic']:.2f}")
            print(f"   ✓ 改进幅度: {improvement:.6f}")
    
    # 创建结果DataFrame
    df_results = pd.DataFrame(results)
    
    # 找到最佳特征组合 (基于交叉验证分数)
    if len(df_results) > 0:
        best_idx = df_results[f'{scoring}_cv_mean'].idxmax()
        best_features = df_results.loc[best_idx, 'features_list']
        best_cv_score = df_results.loc[best_idx, f'{scoring}_cv_mean']
        
        if verbose:
            print(f"\n{'='*60}")
            print(f"特征选择完成!")
            print(f"最佳特征组合 (第{best_idx+1}步): {best_features}")
            print(f"最佳CV {scoring.upper()}分数: {best_cv_score:.6f}")
            print(f"最终选择 {len(best_features)} 个特征")
    else:
        best_features = []
        if verbose:
            print("警告: 没有找到任何有效的特征组合")
    
    return df_results, best_features

def plot_stepwise_results(df_results, scoring='f1'):
    """
    可视化逐步回归结果
    """
    if len(df_results) == 0:
        print("没有结果可视化")
        return
        
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # 1. 评分变化
    axes[0,0].plot(df_results['step'], df_results[f'{scoring}_score'], 
                   'o-', label=f'{scoring.upper()} (训练)', linewidth=2, markersize=6)
    axes[0,0].plot(df_results['step'], df_results[f'{scoring}_cv_mean'], 
                   's-', label=f'{scoring.upper()} (CV)', linewidth=2, markersize=6)
    axes[0,0].fill_between(df_results['step'], 
                          df_results[f'{scoring}_cv_mean'] - df_results[f'{scoring}_cv_std'],
                          df_results[f'{scoring}_cv_mean'] + df_results[f'{scoring}_cv_std'],
                          alpha=0.2)
    axes[0,0].set_xlabel('步数')
    axes[0,0].set_ylabel(f'{scoring.upper()}分数')
    axes[0,0].set_title(f'{scoring.upper()}分数变化')
    axes[0,0].legend()
    axes[0,0].grid(True, alpha=0.3)
    
    # 2. AIC/BIC变化
    axes[0,1].plot(df_results['step'], df_results['aic'], 'o-', label='AIC', linewidth=2)
    axes[0,1].plot(df_results['step'], df_results['bic'], 's-', label='BIC', linewidth=2)
    axes[0,1].set_xlabel('步数')
    axes[0,1].set_ylabel('信息准则')
    axes[0,1].set_title('AIC/BIC变化')
    axes[0,1].legend()
    axes[0,1].grid(True, alpha=0.3)
    
    # 3. 改进幅度
    axes[1,0].bar(df_results['step'], df_results['improvement'], alpha=0.7)
    axes[1,0].set_xlabel('步数')
    axes[1,0].set_ylabel('改进幅度')
    axes[1,0].set_title('每步改进幅度')
    axes[1,0].grid(True, alpha=0.3)
    
    # 4. 特征选择顺序
    feature_names = [f"第{i}步: {feat}" for i, feat in enumerate(df_results['selected_feature'], 1)]
    y_pos = range(len(feature_names))
    
    axes[1,1].barh(y_pos, df_results[f'{scoring}_cv_mean'], alpha=0.7)
    axes[1,1].set_yticks(y_pos)
    axes[1,1].set_yticklabels(feature_names, fontsize=10)
    axes[1,1].set_xlabel(f'CV {scoring.upper()}分数')
    axes[1,1].set_title('特征选择顺序和贡献')
    axes[1,1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# 使用示例和测试
def run_stepwise_example(X, y):
    """
    运行逐步回归示例
    """
    print("=== 前向逐步特征选择示例 ===")
    
    # 确保列名是字符串类型
    X.columns = X.columns.astype(str)
    
    # 运行优化的前向逐步选择
    df_results, best_features = optimized_forward_stepwise(
        y=y, 
        X=X, 
        max_features=8,  # 限制最大特征数
        scoring='f1',    # 使用F1分数
        cv_folds=5,      # 5折交叉验证
        early_stopping=True,
        tolerance=0.001,
        verbose=True
    )
    
    # 显示详细结果
    if len(df_results) > 0:
        print(f"\n=== 详细结果表 ===")
        display_df = df_results.copy()
        display_df['features_list'] = display_df['features_list'].apply(lambda x: ', '.join(x))
        print(display_df.to_string(index=False))
        
        # 可视化结果
        plot_stepwise_results(df_results, scoring='f1')
        
        # 特征重要性分析
        print(f"\n=== 特征选择顺序分析 ===")
        for i, row in df_results.iterrows():
            print(f"第{row['step']:2d}步: 添加 '{row['selected_feature']}' -> "
                  f"F1={row['f1_score']:.4f}, CV_F1={row['f1_cv_mean']:.4f}±{row['f1_cv_std']:.4f}")
        
        return df_results, best_features
    else:
        print("没有找到有效的特征组合")
        return None, []

# 如果有数据，可以直接运行
df_results, best_features = run_stepwise_example(X, y)